## Define job archetype
### Inputs
* Canonical Master Resume (JSON) from stage 2.
* Discovered job descriptions as URLs
### Output
* Target Job archetype (JSON)

    "display_name": "Python 3.12 (ai_py312)",
    "name": "ai_py312"
-  jd_search_contract.json
-  seed_job_descriptions.json

outputs/
-  normalized_jds.json
-  jd_signal_extractions.json
-  canonical_market_signals.json
-  target_archetype.json
-  archetype_coverage_report.csv

In [1]:
%run ./init_notebook.py


Repo root: /Users/douglasdaly/GitHub/Generative-AI
Added src to sys.path: /Users/douglasdaly/GitHub/Generative-AI/src
Resume builder notebooks: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder
Artifacts: /Users/douglasdaly/GitHub/Generative-AI/notebooks/resume-builder/artifacts


In [ ]:
from pathlib import Path
import json
import pandas as pd
from pathlib import Path
from typing import Any
import re
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
from genai_demos.resume_builder.helpers import load_json, save_json
from genai_demos.resume_builder.config import SOURCE_DIR, ARTIFACT_DIR

GENERATE_JD_SIGNAL_EXTRACTIONS = False
GENERATE_NORMALIZED_SIGNAL_GROUPS = False

load_dotenv()
MODEL = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
)


In [2]:
# Load input set
archetype_hypothesis = load_json(SOURCE_DIR / "archetype_hypothesis.json")
jd_search_contract = load_json(SOURCE_DIR / "jd_search_contract.json")
jd_signal_contract = load_json(SOURCE_DIR / "jd_signal_contract.json")
seed_job_descriptions_template = load_json(SOURCE_DIR / "seed_job_descriptions.template.json")

## Phase 3A-0: Define Job Search Strategy

Inputs:
- Candidate job titles
- Target industries

Outputs:
- `sources/target_company_plan.json` -- defines job titles, industries and target companies
- `sources/job_search_strategy.json` -- a set of rules to search for jobs
- `sources/candidate_search_queries.json` -- combination of company plan and job search rules to form broad queries
- `sources/phase3a_search_batch.json` -- aggregate search results across industries

**Note**: this automation is not yet implemented. This is merely the structure. Step 3A-1 uses hand collected JDs.


In [3]:
candidate_job_titles = [
    "Principal AI Engineer",
    "Staff AI Engineer",
    "AI Solutions Architect",
    "Principal AI Architect",
    "Principal Machine Learning Engineer",
    "Staff Machine Learning Engineer",
    "Principal Data Scientist",
    "AI Transformation Lead",
]

target_industries = [
    "enterprise software",
    "cloud platforms",
    "financial services",
    "retail and consumer",
    "media and entertainment",
    "travel and hospitality",
    "consulting",
    "utilities and infrastructure",
    "telecommunications",
]

target_company_plan = {
    "enterprise software": [
        "Salesforce",
        "ServiceNow",
        "Adobe",
        "Datadog",
        "Snowflake",
        "Palantir",
    ],
    "cloud platforms": [
        "AWS",
        "Microsoft",
        "Google",
        "Oracle",
    ],
    "financial services": [
        "Capital One",
        "JPMorgan Chase",
        "American Express",
        "Visa",
        "Mastercard",
        "Fidelity",
        "Charles Schwab",
    ],
    "retail and consumer": [
        "Walmart",
        "Target",
        "Costco",
        "Home Depot",
        "Nike",
        "Starbucks",
        "Best Buy",
    ],
    "media and entertainment": [
        "NBCUniversal",
        "Disney",
        "Netflix",
        "Warner Bros Discovery",
        "Paramount",
    ],
    "travel and hospitality": [
        "Airbnb",
        "Expedia",
        "Booking.com",
        "Hilton",
        "Marriott",
    ],
    "consulting": [
        "Accenture",
        "Deloitte",
        "Slalom",
        "West Monroe",
        "Booz Allen",
        "Capgemini",
        "PwC",
        "EY",
        "KPMG",
    ],
    "utilities and infrastructure": [
        "Consolidated Edison",
        "PG&E",
        "Duke Energy",
        "Southern Company",
        "Schneider Electric",
        "Siemens",
        "GE Vernova",
    ],
    "telecommunications": [
        "Verizon",
        "AT&T",
        "T-Mobile",
        "Comcast",
        "Charter Communications",
    ],
}

save_json(
    target_company_plan,
    SOURCE_DIR / "target_company_plan.json",
)

In [4]:
job_search_strategy = {
    "objective": (
        "Collect a diverse, relevant, text-rich set of job descriptions "
        "for building a target role archetype."
    ),
    "candidate_job_titles": candidate_job_titles,
    "target_industries": target_industries,
    "target_jd_count": 20,
    "minimum_validated_jd_count": 12,
    "collection_rules": [
        "Prefer postings with full responsibilities and qualifications.",
        "Prefer current postings from company career pages or reputable job boards.",
        "Avoid duplicate companies unless the roles represent clearly different archetypes.",
        "Do not treat URLs as sufficient. Raw job-description text is the useful artifact.",
        "Include recruiter-provided descriptions when they are relevant and text-rich.",
    ],
    "diversity_rules": {
        "minimum_industries": 5,
        "max_jds_per_company": 2,
        "max_share_single_title_family": 0.4,
    },
}

save_json(
    job_search_strategy,
    SOURCE_DIR / "job_search_strategy.json",
)

In [5]:
def build_search_queries(job_titles, company_plan):
    queries = []

    for industry, companies in company_plan.items():
        for company in companies:
            for title in job_titles:
                queries.append({
                    "industry": industry,
                    "company": company,
                    "title": title,
                    "query": f'{company} "{title}" careers job description AI',
                    "collection_status": "not_searched",
                })

    return queries


candidate_search_queries = build_search_queries(
    candidate_job_titles,
    target_company_plan,
)

save_json(
    candidate_search_queries,
    SOURCE_DIR / "candidate_search_queries.json",
)

len(candidate_search_queries)

440

In [6]:
def sample_search_queries(candidate_search_queries, max_per_industry=4):
    sampled = []
    counts = {}

    for row in candidate_search_queries:
        industry = row["industry"]
        counts.setdefault(industry, 0)

        if counts[industry] < max_per_industry:
            sampled.append(row)
            counts[industry] += 1

    return sampled


phase3a_search_batch = sample_search_queries(
    candidate_search_queries,
    max_per_industry=4,
)

save_json(
    phase3a_search_batch,
    SOURCE_DIR / "phase3a_search_batch.json",
)

len(phase3a_search_batch)

36

### Phase 3A-0 Output

Created:

- `sources/job_search_strategy.json`
- `sources/target_company_plan.json`
- `sources/candidate_search_queries.json`
- `sources/phase3a_search_batch.json`

These artifacts define where to search. They do not yet contain usable job descriptions.

Next step:

- Search using the query batch.
- Capture job postings with full text.
- Save collected postings to `sources/candidate_job_descriptions.json`.

## Phase 3A-1 - Identify candidate jobs
Use provided URLs first, add automation in future.

In [7]:
candidate_job_descriptions = [
    {
        "title": "Lead Software Engineer - Licensing/AI Systems",
        "company": "Disney",
        "industry": "media and entertainment",
        "source": "Company careers page",
        "url": "https://jobs.disneycareers.com/job/orlando/lead-software-engineer-licensing-ai-systems/391/86564722480",
        "collection_status": "candidate_text_available",
        "notes": "AI-powered applications, intelligent automation, forecasting, compliance monitoring.",
    },
    {
        "title": "Lead Machine Learning Engineer",
        "company": "Disney",
        "industry": "media and entertainment",
        "source": "Company careers page",
        "url": "https://jobs.disneycareers.com/job/new-york/lead-machine-learning-engineer/391/92134437872",
        "collection_status": "candidate_text_available",
        "notes": "Senior IC role for complex ML systems and data foundations.",
    },
    {
        "title": "Director, Decision Science AI/ML Engineering & Ops",
        "company": "Disney",
        "industry": "media and entertainment",
        "source": "Company careers page",
        "url": "https://jobs.disneycareers.com/job/burbank/director-decision-science-ai-ml-engineering-and-ops/391/95735380864",
        "collection_status": "candidate_text_available",
        "notes": "Productionizing decision science, scalable observable resilient AI/ML systems.",
    },
    {
        "title": "Omni-Channel Analytics Manager",
        "company": "Disney",
        "industry": "media and entertainment",
        "source": "Company careers page",
        "url": "https://jobs.disneycareers.com/job/celebration/omni-channel-analytics-mgr/391/95377091536",
        "collection_status": "candidate_text_available",
        "notes": "GenAI, multi-agent systems, MCP, prompt engineering, evaluation frameworks.",
    },
    {
        "title": "Principal Data Scientist, AI Foundations",
        "company": "Capital One",
        "industry": "financial services",
        "source": "Company careers page",
        "url": "https://www.capitalonecareers.com/job/new-york/principal-data-scientist-ai-foundations/1732/86070871904",
        "collection_status": "candidate_text_available",
        "notes": "LLMs, NLP, model lifecycle, scalable production systems.",
    },
    {
        "title": "Distinguished AI Engineer",
        "company": "Capital One",
        "industry": "financial services",
        "source": "Company careers page",
        "url": "https://www.capitalonecareers.com/job/cambridge/distinguished-ai-engineer/1732/93736801200",
        "collection_status": "candidate_text_available",
        "notes": "Foundation models, inference, similarity search, guardrails, evaluation, governance, observability.",
    },
    {
        "title": "Distinguished AI Engineer - Agentic AI Platform",
        "company": "Capital One",
        "industry": "financial services",
        "source": "Company careers page",
        "url": "https://www.capitalonecareers.com/job/san-jose/distinguished-ai-engineer-agentic-ai-platform/1732/89599740832",
        "collection_status": "candidate_text_available",
        "notes": "Agentic workflow framework, memory, guardrails, vector search, SDKs, production-grade applications.",
    },
    {
        "title": "AI Infrastructure Architect",
        "company": "Accenture",
        "industry": "consulting",
        "source": "Company careers page",
        "url": "https://www.accenture.com/us-en/careers/jobdetails?id=ATCI-5197918-S1906698_en",
        "collection_status": "candidate_text_available",
        "notes": "Generative AI, deep learning, neural networks, cross-functional AI delivery.",
    },
    {
        "title": "Advanced AI Architect",
        "company": "Accenture",
        "industry": "consulting",
        "source": "Company careers page",
        "url": "https://www.accenture.com/us-en/careers/jobdetails?id=R00330053_en&title=Advanced+AI+Architect",
        "collection_status": "candidate_text_available",
        "notes": "Full-stack AI architecture, enterprise-grade platforms, security, governance, AI agents.",
    },
    {
        "title": "Data & AI Solution Architect",
        "company": "Accenture",
        "industry": "consulting",
        "source": "Company careers page",
        "url": "https://www.accenture.com/us-en/careers/jobdetails?id=13915760_en",
        "collection_status": "candidate_text_available",
        "notes": "Business objectives, AI/ML opportunities, solution design, GenAI applications.",
    },
    {
        "title": "Software Architect - AI Accelerated Engineering Lead",
        "company": "Slalom",
        "industry": "consulting",
        "source": "Company careers page",
        "url": "https://jobs.slalom.com/en_US/careersmarketplace/JobDetail/Software-Architect-AI-Accelerated-Engineering-Lead-Central/617",
        "collection_status": "candidate_text_available",
        "notes": "Architecture of client solutions and AI-empowered software development.",
    },
    {
        "title": "Senior Consultant - AI Systems & Platforms",
        "company": "Slalom",
        "industry": "consulting",
        "source": "Company careers page",
        "url": "https://jobs.slalom.com/en_US/careersmarketplace/JobDetail?jobId=2548",
        "collection_status": "candidate_text_available",
        "notes": "Designing, building, evaluating, and deploying production AI systems.",
    },
    {
        "title": "Solution Architect - Revenue Management",
        "company": "Marriott",
        "industry": "travel and hospitality",
        "source": "Company careers page",
        "url": "https://careers.marriott.com/solution-architect-revenue-management/job/118B2CD821519877BB640E00DE122BC1",
        "collection_status": "candidate_text_available",
        "notes": "Enterprise architecture for scalable, resilient revenue-management platforms.",
    },
    {
        "title": "Senior Software Engineer - Data and Analytics Technology",
        "company": "Marriott",
        "industry": "travel and hospitality",
        "source": "Company careers page",
        "url": "https://careers.marriott.com/senior-software-engineer-data-and-analytics-technology/job/AECF66C592765E7BBFBA711E750C1051",
        "collection_status": "candidate_text_available",
        "notes": "Technical expert for data platform, reusable frameworks, next-generation data capabilities.",
    },
    {
        "title": "FLEX Sr. Systems Engineer - Observability",
        "company": "Marriott",
        "industry": "travel and hospitality",
        "source": "Company careers page",
        "url": "https://careers.marriott.com/flex-sr-systems-engineer-observability/job/C414CFEF9DAA164DAF82B2817B4B165E",
        "collection_status": "candidate_text_available",
        "notes": "Enterprise observability systems and future-state platform requirements.",
    },
    {
        "title": "Senior, Software Engineer - Full Stack",
        "company": "Walmart",
        "industry": "retail and consumer",
        "source": "Company careers page",
        "url": "https://careers.walmart.com/us/en/jobs/R-2438435",
        "collection_status": "candidate_text_available",
        "notes": "Backend architecture plus exposure to Agentic AI and Generative AI.",
    },
    {
        "title": "Staff Software Engineer - iOS",
        "company": "Walmart",
        "industry": "retail and consumer",
        "source": "Company careers page",
        "url": "https://careers.walmart.com/us/en/jobs/R-2473230",
        "collection_status": "candidate_text_available",
        "notes": "Staff-level technical direction bridging mobile platform work and AI.",
    },
]

In [8]:
save_json(
    candidate_job_descriptions,
    SOURCE_DIR / "candidate_job_descriptions.json",
)

len(candidate_job_descriptions)

17

In [9]:
candidates = load_json(SOURCE_DIR / "candidate_job_descriptions.json")

by_industry = {}
for row in candidates:
    by_industry[row["industry"]] = by_industry.get(row["industry"], 0) + 1

by_industry

{'media and entertainment': 4,
 'financial services': 3,
 'consulting': 5,
 'travel and hospitality': 3,
 'retail and consumer': 2}

In [10]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}


def fetch_url_text(url: str, timeout: int = 20) -> dict:
    result = {
        "status_code": None,
        "final_url": None,
        "fetch_error": None,
        "page_text": "",
    }

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=timeout,
            allow_redirects=True,
        )

        result["status_code"] = response.status_code
        result["final_url"] = response.url

        if not (200 <= response.status_code < 400):
            result["fetch_error"] = f"HTTP {response.status_code}"
            return result

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(separator="\n")
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]+", " ", text)
        text = text.strip()

        result["page_text"] = text

    except Exception as e:
        result["fetch_error"] = str(e)

    return result

In [11]:
RELEVANCE_TERMS = [
    "ai",
    "artificial intelligence",
    "machine learning",
    "ml",
    "llm",
    "generative ai",
    "genai",
    "data science",
    "model",
    "architecture",
    "architect",
    "production",
    "platform",
    "cloud",
    "rag",
    "agent",
]

JD_STRUCTURE_TERMS = [
    "job description",
    "project role",
    "project role description",
    "summary",
    "roles & responsibilities",
    "roles and responsibilities",
    "responsibilities",
    "professional & technical skills",
    "professional and technical skills",
    "technical skills",
    "must have skills",
    "good to have skills",
    "skills and qualifications",
    "qualifications",
    "requirements",
    "minimum qualifications",
    "preferred qualifications",
    "additional information",
]

QUALIFICATION_TERMS = [
    "qualifications",
    "skills and qualifications",
    "requirements",
    "required skills",
    "basic qualifications",
    "minimum qualifications",
    "preferred qualifications",
    "what you'll bring",
    "what you bring",
    "who you are",
]

RESPONSIBILITY_TERMS = [
    "responsibilities",
    "responsibility",
    "what you'll do",
    "what you will do",
    "duties",
    "role responsibilities",
    "key responsibilities",
    "your role",
    "the work",
    "job description",
    "about the role",
]

def looks_like_search_page(text: str) -> bool:
    text_lower = text.lower()

    return (
        "search jobs" in text_lower
        and "job alerts" in text_lower
        and text_lower.count("apply") > 10
    )


def validate_jd_text(text: str) -> dict:
    text_lower = text.lower()
    words = text.split()

    relevance_hits = sorted({
        term for term in RELEVANCE_TERMS
        if term in text_lower
    })

    structure_hits = sorted({
        term for term in JD_STRUCTURE_TERMS
        if term in text_lower
    })

    word_count = len(words)

    is_text_rich = word_count >= 350
    is_relevant = len(relevance_hits) >= 3
    has_jd_structure = len(structure_hits) >= 2

    is_usable = (
        is_text_rich
        and is_relevant
        and has_jd_structure
    )

    return {
        "word_count": word_count,
        "relevance_hits": relevance_hits,
        "structure_hits": structure_hits,
        "is_text_rich": is_text_rich,
        "is_relevant": is_relevant,
        "has_jd_structure": has_jd_structure,
        "is_usable": is_usable,
    }

In [12]:
validated_jds = []

for jd in candidates:
    fetched = fetch_url_text(jd["url"])

    raw_text = fetched["page_text"]

    validation = validate_jd_text(raw_text)

    validated_jds.append({
        **jd,
        "raw_text": raw_text,
        "fetch": {
            "status_code": fetched["status_code"],
            "final_url": fetched["final_url"],
            "fetch_error": fetched["fetch_error"],
        },
        "validation": validation,
    })

save_json(
    validated_jds,
    ARTIFACT_DIR / "validated_job_descriptions.json",
)

sum(row["validation"]["is_usable"] for row in validated_jds), len(validated_jds)

(15, 17)

In [13]:
# Inspect why validation failed

diagnostics = []

for row in validated_jds:
    diagnostics.append({
        "company": row["company"],
        "title": row["title"],
        "url": row["url"],
        "status_code": row["fetch"]["status_code"],
        "fetch_error": row["fetch"]["fetch_error"],
        "word_count": row["validation"]["word_count"],
        "relevance_hits": row["validation"]["relevance_hits"],
        "is_text_rich": row["validation"]["is_text_rich"],
        "is_relevant": row["validation"]["is_relevant"],
        "has_jd_structure": row["validation"]["has_jd_structure"],
        "is_usable": row["validation"]["is_usable"],
        "text_preview": row["raw_text"][:500],
    })

diagnostics_df = pd.DataFrame(diagnostics)

diagnostics_df[
    [
        "company",
        "title",
        "status_code",
        "word_count",
        "is_text_rich",
        "is_relevant",
        "has_jd_structure",
        "is_usable",
        "fetch_error",
        "text_preview",
    ]
].sort_values("word_count", ascending=False)

,company,title,status_code,word_count,is_text_rich,is_relevant,has_jd_structure,is_usable,fetch_error,text_preview
2,Disney,"Director, Decision Science AI/ML Engineering &...",200,3138,True,True,True,True,None,"Director, Decision Science AI/ML Engineering &..."
6,Capital One,Distinguished AI Engineer - Agentic AI Platform,200,2496,True,True,True,True,None,Distinguished AI Engineer (Agentic AI Platform...
1,Disney,Lead Machine Learning Engineer,200,2364,True,True,True,True,None,Lead Machine Learning Engineer at DISNEY\n\nSk...
4,Capital One,"Principal Data Scientist, AI Foundations",200,2315,True,True,True,True,None,"Principal Data Scientist, AI Foundations at Ca..."
5,Capital One,Distinguished AI Engineer,200,2301,True,True,True,True,None,Distinguished AI Engineer at Capital One\n\nSk...
0,Disney,Lead Software Engineer - Licensing/AI Systems,200,2116,True,True,True,True,None,Lead Software Engineer-Licensing/AI Systems at...
8,Accenture,Advanced AI Architect,200,1953,True,True,True,True,None,Advanced AI Architect\n\nSkip to main content\...
14,Marriott,FLEX Sr. Systems Engineer - Observability,200,1907,True,True,True,True,None,FLEX Sr. Systems Engineer - Observability | Ma...
12,Marriott,Solution Architect - Revenue Management,200,1768,True,True,True,True,None,Solution Architect – Revenue Management | Marr...
13,Marriott,Senior Software Engineer - Data and Analytics ...,200,1675,True,True,True,True,None,Senior Software Engineer – Data and Analytics ...


In [14]:
diagnostics_df.to_csv('diagnostics.csv')

In [15]:
failed = diagnostics_df[~diagnostics_df["is_usable"]]

failed[
    [
        "company",
        "title",
        "status_code",
        "word_count",
        "is_text_rich",
        "is_relevant",
        "has_jd_structure",
        "fetch_error",
        "text_preview",
    ]
]

,company,title,status_code,word_count,is_text_rich,is_relevant,has_jd_structure,fetch_error,text_preview
3,Disney,Omni-Channel Analytics Manager,404,0,False,False,False,HTTP 404,
16,Walmart,Staff Software Engineer - iOS,200,219,False,False,False,None,Skip to main content\nCareer areas\nBrands\nRe...


In [16]:
import pandas as pd

validation_df = pd.DataFrame([
    {
        "company": row["company"],
        "title": row["title"],
        "industry": row.get("industry", ""),
        "status_code": row["fetch"]["status_code"],
        "word_count": row["validation"]["word_count"],
        "is_text_rich": row["validation"]["is_text_rich"],
        "is_relevant": row["validation"]["is_relevant"],
        "has_jd_structure": row["validation"]["has_jd_structure"],
        "is_usable": row["validation"]["is_usable"],
        "fetch_error": row["fetch"]["fetch_error"],
        "final_url": row["fetch"]["final_url"],
    }
    for row in validated_jds
])

validation_df.sort_values(
    ["is_usable", "word_count"],
    ascending=[False, False],
)

,company,title,industry,status_code,word_count,is_text_rich,is_relevant,has_jd_structure,is_usable,fetch_error,final_url
2,Disney,"Director, Decision Science AI/ML Engineering &...",media and entertainment,200,3138,True,True,True,True,None,https://jobs.disneycareers.com/job/burbank/dir...
6,Capital One,Distinguished AI Engineer - Agentic AI Platform,financial services,200,2496,True,True,True,True,None,https://www.capitalonecareers.com/job/san-jose...
1,Disney,Lead Machine Learning Engineer,media and entertainment,200,2364,True,True,True,True,None,https://jobs.disneycareers.com/job/new-york/le...
4,Capital One,"Principal Data Scientist, AI Foundations",financial services,200,2315,True,True,True,True,None,https://www.capitalonecareers.com/job/new-york...
5,Capital One,Distinguished AI Engineer,financial services,200,2301,True,True,True,True,None,https://www.capitalonecareers.com/job/cambridg...
0,Disney,Lead Software Engineer - Licensing/AI Systems,media and entertainment,200,2116,True,True,True,True,None,https://jobs.disneycareers.com/job/orlando/lea...
8,Accenture,Advanced AI Architect,consulting,200,1953,True,True,True,True,None,https://www.accenture.com/us-en/careers/jobdet...
14,Marriott,FLEX Sr. Systems Engineer - Observability,travel and hospitality,200,1907,True,True,True,True,None,https://careers.marriott.com/flex-sr-systems-e...
12,Marriott,Solution Architect - Revenue Management,travel and hospitality,200,1768,True,True,True,True,None,https://careers.marriott.com/solution-architec...
13,Marriott,Senior Software Engineer - Data and Analytics ...,travel and hospitality,200,1675,True,True,True,True,None,https://careers.marriott.com/senior-software-e...


In [17]:
usable_jds = [
    row for row in validated_jds
    if row["validation"]["is_usable"]
]

save_json(
    usable_jds,
    ARTIFACT_DIR / "usable_job_descriptions.json",
)

len(usable_jds)

15

## Phase 3B: extract JD signals from the 15 usable job descriptions.

Inputs:

artifacts/usable_job_descriptions.json
sources/jd_signal_contract.json

Output:

artifacts/jd_signal_extractions.json

In [ ]:
from langchain_openai import ChatOpenAI
from genai_demos.resume_builder.capabilities import call_json_model

load_dotenv()
MODEL = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
)

usable_jds = load_json(ARTIFACT_DIR / "usable_job_descriptions.json")
jd_signal_contract = load_json(SOURCE_DIR / "jd_signal_contract.json")

len(usable_jds)


15

In [ ]:
from genai_demos.resume_builder.contracts import JD_SIGNAL_EXTRACTION_SCHEMA


def build_jd_signal_prompt(jd, jd_signal_contract):
    return f"""
You are extracting structured signals from a job description.

Return ONLY valid JSON.
Do not include markdown.
Do not include explanations.

Signal contract:
{json.dumps(jd_signal_contract, indent=2)}

Job description:
{json.dumps(jd, indent=2)}

Return JSON matching this schema:

{json.dumps(JD_SIGNAL_EXTRACTION_SCHEMA, indent=2)}
"""


In [20]:
if GENERATE_JD_SIGNAL_EXTRACTIONS:
    jd_signal_extractions = []

    for jd in usable_jds:
        prompt = build_jd_signal_prompt(jd, jd_signal_contract)
        result = call_json_model(prompt, model=MODEL)
        jd_signal_extractions.append(result)

    save_json(
        jd_signal_extractions,
        ARTIFACT_DIR / "jd_signal_extractions.json",
    )
else:
    jd_signal_extractions = load_json(ARTIFACT_DIR / "jd_signal_extractions.json")
len(jd_signal_extractions)

15

In [21]:
from collections import Counter
check_dims = ['technology_areas', 'capabilities', 'problem_spaces', 'seniority_signals']

summary_df = []
for dim in check_dims:
    counter = Counter()

    for jd in jd_signal_extractions:
        counter.update(
            jd["signals"][dim]
        )
    cur_df = pd.DataFrame(data=counter.most_common(), columns=['type','count'])
    cur_df['dim'] = dim
    summary_df.append(cur_df)

result = pd.concat(summary_df)[['dim','type','count']]
result.to_csv('jd_signal_counts.csv', index=False)
display(result)

,dim,type,count
0,technology_areas,MLOps,6
1,technology_areas,cloud-native AI systems,5
2,technology_areas,LLMs,4
3,technology_areas,agentic AI,4
4,technology_areas,deep learning,3
...,...,...,...
109,seniority_signals,lead the design and implementation,1
110,seniority_signals,mentoring team members,1
111,seniority_signals,driving best practices,1
112,seniority_signals,collaborate closely with architects and staff ...,1


### Stage 3C: Generate Canonical Market Signals
Purpose:
Convert raw JD signal extracts into a stable market vocabulary
that can be used to generate the target career archetype.

Input:
artifacts/jd_signal_extractions.json

- Job description signal extracts
- One extracted signal set per validated job description
- Conforms to JD signal extraction schema


Output:
artifacts/canonical_market_signals.json

- Normalized market signals grouped by signal type
- Semantically equivalent signals consolidated into canonical signals
- Includes market evidence such as:
    - phrase_count
    - jd_coverage
    - companies
    - source_phrases
    - example_phrases
- Conforms to Canonical Market Signal schema


In [22]:
CANONICAL_SIGNAL_LIMITS = {
    "business_industries": 12,
    "problem_spaces": 15,
    "technology_areas": 15,
    "capabilities": 20,
    "technologies": 25,
    "responsibilities": 15,
    "seniority_signals": 12,
    "success_metrics": 12,
    "role_purpose": 10,
    "business_problems": 12,
    "role_outcomes": 12,
    "emerging_signals": 12,
}

In [ ]:
from collections import defaultdict
import json

from genai_demos.resume_builder.helpers import load_json, save_json
from genai_demos.resume_builder.contracts import JD_SIGNAL_FIELDS

jd_signal_extractions = load_json(ARTIFACT_DIR / "jd_signal_extractions.json")


In [24]:

from collections import Counter, defaultdict
import json

def collect_unique_phrases_by_type(jd_signal_extractions):
    phrases_by_type = defaultdict(Counter)

    for jd in jd_signal_extractions:
        for signal_type in JD_SIGNAL_FIELDS:
            for phrase in jd.get("signals", {}).get(signal_type, []):
                phrase = str(phrase).strip()
                if phrase:
                    phrases_by_type[signal_type][phrase] += 1

    return dict(phrases_by_type)

unique_phrases_by_type = collect_unique_phrases_by_type(jd_signal_extractions)

In [25]:
# Raw signals is for evidence -- proof that a signal type exists in a JD
# Unique phrases is for normalization -- categorize phrases into groups

def collect_raw_signals_by_type(jd_signal_extractions):
    signals_by_type = defaultdict(list)

    for jd in jd_signal_extractions:
        company = jd.get("company", "")
        title = jd.get("title", "")
        url = jd.get("url", "")

        for signal_type in JD_SIGNAL_FIELDS:
            for phrase in jd.get("signals", {}).get(signal_type, []):
                phrase = str(phrase).strip()
                if not phrase:
                    continue

                signals_by_type[signal_type].append({
                    "phrase": phrase,
                    "company": company,
                    "title": title,
                    "url": url,
                })

    return dict(signals_by_type)


raw_signals_by_type = collect_raw_signals_by_type(jd_signal_extractions)


In [26]:
def build_signal_normalization_prompt(signal_type, phrase_counter, max_signals):
    phrases = [
        {"phrase": phrase, "count": count}
        for phrase, count in phrase_counter.most_common(200)
    ]

    return f"""
Normalize market signals for one signal type.

Signal type:
{signal_type}

Group semantically similar phrases into canonical market signals.

Rules:
- Work only within this signal type.
- Use at most {max_signals} canonical signals.
- Prefer snake_case canonical signal names.
- Keep source_phrases exactly as provided.
- Return valid JSON only.
- No markdown.

Raw phrases:
{json.dumps(phrases, indent=2)}

Return:
{{
  "signal_type": "{signal_type}",
  "canonical_groups": [
    {{
      "canonical_signal": "snake_case_name",
      "source_phrases": ["exact source phrase"],
      "rationale": "brief rationale"
    }}
  ]
}}
"""

In [27]:
if GENERATE_NORMALIZED_SIGNAL_GROUPS:
    normalized_signal_groups = {}

    for signal_type, phrase_counter in unique_phrases_by_type.items():
        max_signals = CANONICAL_SIGNAL_LIMITS.get(signal_type, 12)

        print("Normalizing:", signal_type, "unique phrases:", len(phrase_counter))

        normalized_signal_groups[signal_type] = call_json_model(
            build_signal_normalization_prompt(
                signal_type,
                phrase_counter,
                max_signals,
            ),
            model=MODEL,
        )

    save_json(
        normalized_signal_groups,
        ARTIFACT_DIR / "normalized_signal_groups.json",
    )
else:
    normalized_signal_groups = load_json(ARTIFACT_DIR / "normalized_signal_groups.json")

In [28]:
def build_canonical_market_signals(raw_signals_by_type, normalized_signal_groups):
    output = {}

    for signal_type, normalized in normalized_signal_groups.items():
        raw_records = raw_signals_by_type[signal_type]
        canonical_entries = []

        for group in normalized["canonical_groups"]:
            source_phrases = set(group["source_phrases"])

            matching_records = [
                row for row in raw_records
                if row["phrase"] in source_phrases
            ]

            companies = sorted({
                row["company"]
                for row in matching_records
                if row.get("company")
            })

            jd_keys = sorted({
                f"{row.get('company', '')} | {row.get('title', '')}"
                for row in matching_records
            })

            canonical_entries.append({
                "signal_type": signal_type,
                "canonical_signal": group["canonical_signal"],
                "phrase_count": len(matching_records),
                "jd_coverage": len(jd_keys),
                "companies": companies,
                "source_phrases": sorted(source_phrases),
                "example_phrases": sorted(source_phrases)[:5],
                "rationale": group.get("rationale", ""),
            })

        canonical_entries = sorted(
            canonical_entries,
            key=lambda x: (x["jd_coverage"], x["phrase_count"]),
            reverse=True,
        )

        output[signal_type] = canonical_entries

    return output


canonical_market_signals = build_canonical_market_signals(
    raw_signals_by_type,
    normalized_signal_groups,
)

save_json(
    canonical_market_signals,
    ARTIFACT_DIR / "canonical_market_signals.json",
)

In [29]:
for signal_type, rows in canonical_market_signals.items():
    print("\n" + signal_type.upper())
    for row in rows[:10]:
        print(
            row["canonical_signal"],
            "| jd_coverage:",
            row["jd_coverage"],
            "| phrase_count:",
            row["phrase_count"],
        )


BUSINESS_INDUSTRIES
technology_and_software | jd_coverage: 11 | phrase_count: 16
financial_services | jd_coverage: 8 | phrase_count: 22
travel_and_hospitality | jd_coverage: 8 | phrase_count: 9
consumer_goods_and_retail | jd_coverage: 7 | phrase_count: 12
media_and_entertainment | jd_coverage: 6 | phrase_count: 12
public_sector_and_government | jd_coverage: 5 | phrase_count: 8
industrial_and_manufacturing | jd_coverage: 4 | phrase_count: 13
healthcare_and_life_sciences | jd_coverage: 4 | phrase_count: 9
consulting_and_professional_services | jd_coverage: 4 | phrase_count: 4
energy_and_natural_resources | jd_coverage: 3 | phrase_count: 9

PROBLEM_SPACES
decision_support_and_analytics | jd_coverage: 9 | phrase_count: 11
workflow_automation | jd_coverage: 9 | phrase_count: 10
observability_and_monitoring | jd_coverage: 7 | phrase_count: 10
risk_and_compliance | jd_coverage: 7 | phrase_count: 8
knowledge_and_information_retrieval | jd_coverage: 6 | phrase_count: 6
ai_enablement_and_adopti

### Stage 3C Closeout

Stage 3C generated `artifacts/canonical_market_signals.json`.

This artifact consolidates raw JD signal extractions into normalized market signals grouped by signal type. The strongest market themes are:

- AI architecture and design
- Production AI / ML systems
- Workflow automation
- Decision support and analytics
- Cloud-native AI platforms
- MLOps / LLMOps
- Agentic AI and LLM systems
- Observability, reliability, governance, and compliance
- Cross-functional communication and technical leadership
- Mentorship and team enablement

The signal extraction and aggregation process is good enough to support archetype generation.

Caveat: business industry signals are useful for breadth but should be weighted lightly because consulting JDs often enumerate many served industries.

## Stage 3D: Generate Target Archetype

Input:
- `artifacts/canonical_market_signals.json`
- `sources/archetype_hypothesis.json`

Output:
- `artifacts/target_archetype.json`

Purpose:
Generate a market-grounded target archetype for the resume. The archetype should use the hypothesis as a starting point, but the canonical market signals should determine what is confirmed, revised, added, or downweighted.

In [30]:
canonical_market_signals = load_json(
    ARTIFACT_DIR / "canonical_market_signals.json"
)

archetype_hypothesis = load_json(
    SOURCE_DIR / "archetype_hypothesis.json"
)

In [ ]:
from genai_demos.resume_builder.contracts import TARGET_ARCHETYPE_SCHEMA

def build_target_archetype_prompt(
    canonical_market_signals,
    archetype_hypothesis,
    target_archetype_schema,
    jd_count,
):
    return f"""
Generate a target career archetype from market signals.

Use the archetype hypothesis as a starting point, not as the answer.
The canonical market signals are the primary evidence.

Facts:
- Number of validated job descriptions analyzed: {jd_count}

Rules:
- Ground the archetype in the market signals.
- Confirm, revise, downweight, or add to the hypothesis as needed.
- Weight capabilities, technology areas, problem spaces, seniority signals, role purpose, and success metrics most heavily.
- Weight business industries and technologies more lightly.
- Do not overfit to any one company.
- Do not include every signal. Select the most important recurring and strategically relevant themes.
- Return valid JSON only.
- No markdown.

Archetype hypothesis:
{json.dumps(archetype_hypothesis, indent=2)}

Canonical market signals:
{json.dumps(canonical_market_signals, indent=2)}

Return JSON matching this schema:
{json.dumps(target_archetype_schema, indent=2)}
"""


In [39]:
jd_count = len(usable_jds)

prompt = build_target_archetype_prompt(
    canonical_market_signals=canonical_market_signals,
    archetype_hypothesis=archetype_hypothesis,
    target_archetype_schema=TARGET_ARCHETYPE_SCHEMA,
    jd_count=jd_count,
)

target_archetype = call_json_model(
    prompt,
    model=MODEL,
)
target_archetype["market_basis"]["jd_count"] = jd_count

save_json(
    target_archetype,
    ARTIFACT_DIR / "target_archetype.json",
)

target_archetype

{'title': 'Principal AI Architect / AI Platform Engineering Lead',
 'archetype_summary': 'A senior technical leader responsible for architecting, delivering, and scaling enterprise-grade AI/ML platforms and solutions. This role bridges business needs and technical execution, with a strong focus on productionizing generative and agentic AI, ensuring operational reliability, governance, and cross-functional enablement. The archetype is defined by deep expertise in AI architecture, cloud-native systems, MLOps, and technical leadership, with a mandate to drive innovation, adoption, and measurable business value at scale.',
 'market_basis': {'jd_count': 15,
  'primary_signal_types_used': ['capabilities',
   'technology_areas',
   'problem_spaces',
   'seniority_signals',
   'role_purpose',
   'success_metrics',
   'responsibilities',
   'role_outcomes'],
  'notes': 'Analysis is grounded in recurring, high-weight signals across multiple industries and companies, with particular emphasis on c

In [40]:
target_archetype["archetype_summary"]

'A senior technical leader responsible for architecting, delivering, and scaling enterprise-grade AI/ML platforms and solutions. This role bridges business needs and technical execution, with a strong focus on productionizing generative and agentic AI, ensuring operational reliability, governance, and cross-functional enablement. The archetype is defined by deep expertise in AI architecture, cloud-native systems, MLOps, and technical leadership, with a mandate to drive innovation, adoption, and measurable business value at scale.'

In [41]:
target_archetype["resume_implications"]

{'must_emphasize': ['AI/ML platform and solution architecture at enterprise scale',
  'Productionizing generative/agentic AI and LLMs',
  'Cloud-native system design and MLOps/LLMOps expertise',
  'Technical leadership, mentorship, and cross-functional influence',
  'Operational reliability, governance, and responsible AI'],
 'should_include': ['Reusable frameworks, developer tools, and enablement assets',
  'Business alignment and stakeholder communication',
  'Continuous learning, research application, and innovation',
  'Experience with observability, monitoring, and compliance',
  'Delivery of measurable business outcomes and customer impact'],
 'use_lightly': ['Industry-specific experience (unless highly relevant to target employer)',
  'Niche technologies or problem spaces (e.g., graph analytics, entity resolution) unless central to recent roles'],
 'avoid_overemphasizing': ['Single-company or single-industry specialization',
  'Legacy-only technology stacks without evidence of m

## Build career archetype from target resume

In [ ]:
def build_career_archetype(master_resume: dict, archetype_schema: dict, model: str = MODEL) -> dict:
    prompt = f"""
Analyze this master resume and extract the candidate's career archetype.

Use the same schema as the market archetype.

Rules:
- Use only evidence from the master resume.
- Do not invent experience, tools, industries, metrics, employers, or dates.
- Prefer broad reusable signals over overly specific bullet details.
- Include differentiators that may not appear in the market archetype.
- Return only valid JSON.
- Do not include markdown fences.

Return JSON using this schema shape:

{json.dumps(archetype_schema, indent=2)}

Master Resume:
{json.dumps(master_resume, indent=2)}
"""

    response = client.responses.create(
        model=model,
        input=prompt,
        temperature=0,
    )

    return json.loads(response.output_text)

In [ ]:
# Load master resume and build into career archetype
with open(ARTIFACT_DIR / "master_resume.json") as f:
    master_resume = json.load(f)

CAREER_ARCHETYPE_V0 = build_career_archetype(
    master_resume=master_resume,
    archetype_schema=MARKET_ARCHETYPE_V0,
)

with open("artifacts/career_archetype_v0.json", "w") as f:
    json.dump(CAREER_ARCHETYPE_V0, f, indent=2)

In [ ]:
# Analyze fit between master resume and the career archetype
def compare_market_to_career(market_archetype, career_archetype, model=MODEL):
    prompt = f"""
Compare the market archetype against the candidate career archetype.

Task:
Identify strong matches, partial matches, differentiators, and gaps.

Rules:
- Use only the two archetypes provided.
- Do not invent candidate experience.
- Treat market archetype as the target.
- Treat career archetype as evidence from the resume.
- Distinguish between true gaps and messaging gaps.
- Return only valid JSON.
- Do not include markdown fences.

Return JSON in this format:

{{
  "strong_matches": [],
  "partial_matches": [],
  "candidate_differentiators": [],
  "true_gaps": [],
  "messaging_gaps": [],
  "resume_strategy": [],
  "notes": []
}}

Market Archetype:
{json.dumps(market_archetype, indent=2)}

Career Archetype:
{json.dumps(career_archetype, indent=2)}
"""
    response = client.responses.create(
        model=model,
        input=prompt,
        temperature=0,
    )

    return json.loads(response.output_text)

In [ ]:
job_match = compare_market_to_career(MARKET_ARCHETYPE_V0, CAREER_ARCHETYPE_V0, model=MODEL)

In [ ]:
job_match

{'strong_matches': ['title',
  'business_industries: enterprise software, financial services, marketing technology, digital advertising, regulated industries',
  'problem_spaces: observability and monitoring, enterprise data integration',
  'technology_areas: LLMs, RAG, agentic AI, cloud-native AI systems, MLOps, distributed systems, multi-agent systems',
  'capabilities: design AI architectures, integrate AI systems with enterprise data, orchestrate AI agents and tools, evaluate AI models, manage monitoring and observability, lead technical roadmaps, communicate technical tradeoffs, mentor engineers, deliver production-grade systems',
  'technologies: Python, AWS, GCP, MCP servers, vector and graph databases, CI/CD',
  'responsibilities: lead end-to-end AI solution development, design scalable AI architectures, integrate AI agents with enterprise systems, define architecture standards, implement monitoring and observability, govern tool orchestration, collaborate with stakeholders, me

In [ ]:
# Strategy from Chat GPT after reading the fit analysis
RESUME_STRATEGY_V0 = {
    "target_title": "Principal AI Engineer / AI Solutions Architect",
    "core_positioning": "Enterprise AI architect with deep operational intelligence, observability, and decision-support background.",
    "must_emphasize": [
        "production AI systems",
        "RAG and agentic AI",
        "enterprise data integration",
        "AI system reliability and observability",
        "technical roadmap and architecture ownership",
        "stakeholder communication",
    ],
    "differentiators": [
        "operational intelligence",
        "graph analytics",
        "AI governance",
        "patented systems work",
        "cross-industry decision-support systems",
        "AI teaching and enablement",
    ],
    "avoid_overclaiming": [
        "Kubernetes",
        "scalable inference systems",
        "foundation model training",
    ],
    "rewrite_priorities": [
        "Make ConEd and NBCU carry the AI architect narrative.",
        "Frame Meta, Capital One, Facteus, and Raytheon as the deeper operational-intelligence backbone.",
        "Use older analytics work as evidence of decision systems, not as a separate data-science story.",
    ],
}

## Stage 3 Closeout

Stage 3 established a market-grounded target career archetype.

The workflow collected, validated, and analyzed representative job descriptions from the target market. Raw job description content was transformed into structured signal extractions, normalized into canonical market signals, and synthesized into a target archetype.

Artifacts produced:

- artifacts/usable_job_descriptions.json
- artifacts/jd_signal_extractions.json
- artifacts/canonical_market_signals.json
- artifacts/target_archetype.json

Key findings:

- The target market strongly emphasizes enterprise AI architecture and production AI systems.
- Generative AI, agentic AI, MLOps/LLMOps, governance, observability, and platform engineering are recurring themes.
- Technical leadership, mentorship, cross-functional influence, and business translation are consistently valued.
- Industry experience is secondary to architecture, platform, and delivery capabilities.
- The candidate's background aligns strongly with the market archetype, with several differentiators beyond the baseline market expectations.

Stage 4 will shift focus from the market to the candidate.

Inputs:
- artifacts/canonical_master_resume.json
- artifacts/target_archetype.json

Objective:
Evaluate and score candidate evidence against the target archetype to determine which accomplishments most strongly support the target market positioning.